<a href="https://colab.research.google.com/github/SuhasGowdaN-ABHI/CIT-PROJECT/blob/main/airesume.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🤖 AI RESUME ANALYZER - GOOGLE COLAB
# GenAI + Gradio + PDF/DOCX Resume Analysis
# ============================================================

# Install required libraries
!pip -q install -U google-genai gradio pypdf python-docx

import os
import json
import re
import gradio as gr
from pypdf import PdfReader
from docx import Document
from google import genai

# ============================================================
# 1. ENTER YOUR GEMINI API KEY
# ============================================================

API_KEY = input("🔑 Enter your Gemini API Key: ")

client = genai.Client(api_key=API_KEY)

# ============================================================
# 2. EXTRACT TEXT FROM PDF
# ============================================================

def extract_pdf_text(file_path):
    text = ""

    reader = PdfReader(file_path)

    for page in reader.pages:
        page_text = page.extract_text()

        if page_text:
            text += page_text + "\n"

    return text


# ============================================================
# 3. EXTRACT TEXT FROM DOCX
# ============================================================

def extract_docx_text(file_path):
    doc = Document(file_path)

    text = ""

    for paragraph in doc.paragraphs:
        text += paragraph.text + "\n"

    return text


# ============================================================
# 4. EXTRACT RESUME TEXT
# ============================================================

def extract_resume_text(file_path):

    if file_path is None:
        return ""

    extension = os.path.splitext(file_path)[1].lower()

    if extension == ".pdf":
        return extract_pdf_text(file_path)

    elif extension == ".docx":
        return extract_docx_text(file_path)

    else:
        return "Unsupported file format."


# ============================================================
# 5. GENAI RESUME ANALYZER
# ============================================================

def analyze_resume(file_path, target_job):

    if file_path is None:
        return "❌ Please upload your resume."

    resume_text = extract_resume_text(file_path)

    if not resume_text.strip():
        return "❌ Could not extract text from the resume."

    # Limit extremely large resumes
    resume_text = resume_text[:30000]

    prompt = f"""
You are an expert AI Resume Analyzer and ATS system.

Analyze the following resume for the target job role:

TARGET JOB:
{target_job}

RESUME:
{resume_text}

Give the analysis in a clear professional format.

Include:

1. ATS SCORE
Give a score from 0 to 100.

2. RESUME SUMMARY
Give a short summary of the candidate.

3. DETECTED SKILLS
Separate into:
- Technical Skills
- Soft Skills
- Tools/Technologies

4. MISSING SKILLS
List important skills missing for the target job.

5. EXPERIENCE ANALYSIS
Explain whether the experience matches the target job.

6. EDUCATION ANALYSIS

7. PROJECT ANALYSIS
Explain the strength of projects.

8. RESUME STRENGTHS
Give at least 5 strengths if possible.

9. RESUME WEAKNESSES
Give at least 5 weaknesses if possible.

10. IMPROVEMENTS
Give specific improvements section by section.

11. KEYWORD SUGGESTIONS
Give important ATS keywords that should be added naturally.

12. JOB MATCH
Give a percentage showing how well the resume matches the target job.

13. INTERVIEW PREPARATION
Give 5 important topics the candidate should prepare.

14. FINAL VERDICT
Explain whether the candidate is:
- Beginner
- Intermediate
- Job Ready

Be honest. Do not invent experience, education, or skills.
"""

    try:

        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        return response.text

    except Exception as e:

        return f"❌ Error while analyzing resume:\n\n{str(e)}"


# ============================================================
# 6. CREATE WEBSITE UI
# ============================================================

custom_css = """
body {
    background: #f5f7fb;
}

.gradio-container {
    max-width: 1100px !important;
    margin: auto;
}

.title {
    text-align: center;
    font-size: 38px;
    font-weight: bold;
}

.subtitle {
    text-align: center;
    font-size: 18px;
    color: #555;
}

.result-box {
    font-size: 16px;
}
"""


with gr.Blocks(
    title="AI Resume Analyzer",
    css=custom_css,
    theme=gr.themes.Soft()
) as app:

    gr.Markdown(
        """
        <div class="title">
        🤖 AI Resume Analyzer
        </div>

        <div class="subtitle">
        GenAI-powered Resume Analysis & ATS Optimization
        </div>
        """
    )

    gr.Markdown(
        """
        ### 🚀 How it works

        **Upload Resume → Select Job Role → AI Analysis → ATS Score → Skills → Improvements**
        """
    )

    with gr.Row():

        with gr.Column():

            resume_file = gr.File(
                label="📄 Upload Resume",
                file_types=[".pdf", ".docx"],
                type="filepath"
            )

            job_role = gr.Dropdown(
                choices=[
                    "Software Developer",
                    "Python Developer",
                    "Java Developer",
                    "Web Developer",
                    "Frontend Developer",
                    "Backend Developer",
                    "Full Stack Developer",
                    "Data Analyst",
                    "Data Scientist",
                    "Machine Learning Engineer",
                    "AI Engineer",
                    "Generative AI Engineer",
                    "DevOps Engineer",
                    "Cybersecurity Engineer"
                ],
                label="💼 Target Job Role",
                value="Software Developer",
                allow_custom_value=True
            )

            analyze_button = gr.Button(
                "🚀 Analyze My Resume",
                variant="primary"
            )

        with gr.Column():

            output = gr.Markdown(
                label="📊 AI Analysis",
                elem_classes=["result-box"]
            )

    analyze_button.click(
        fn=analyze_resume,
        inputs=[resume_file, job_role],
        outputs=output
    )

    gr.Markdown(
        """
        ---
        ### 🧠 AI Resume Analyzer

        Built with:
        **Python + Google Gemini + Gradio + NLP + ATS Analysis**

        ⚠️ Do not upload resumes containing sensitive information you don't want processed.
        """
    )


# ============================================================
# 7. LAUNCH WEBSITE
# ============================================================

app.launch(
    share=True,
    debug=True
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 12.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.
🔑 Enter your Gemini API Key: AQ.Ab8RN6IHrUx9wc9XOysFoqZapcvZmF1StD_xb2RqmfOAI9VofAQ.Ab8RN6IHrUx9wc9XOysFoqZapcvZmF1StD_xb2RqmfOAI9Vof


/tmp/ipykernel_539/2374563252.py:207: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2ae4bf0a70ba99a840.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
